# Module 3 Lab 2: Textbook Problem 2.1

**CS 82A - Introduction to Data Science**

**Ramisha Tasfia**

**GitHub:** https://github.com/ramishatasfia5156-commits/cs82a-coursework/blob/main/module3/module3_lab2.ipynb

In [1]:
import pandas as pd

df = pd.read_csv('usarrests.csv', index_col=0)
df.index.name = 'State'

print('Shape:', df.shape)
print()
print('Missing values per column:')
print(df.isna().sum())

Shape: (50, 3)

Missing values per column:
Murder      0
Assault     1
UrbanPop    0
dtype: int64


## Summary Statistics

In [2]:
df.describe().T[['min', '25%', '50%', 'mean', '75%', 'max']].round(2)

,min,25%,50%,mean,75%,max
Murder,0.8,4.08,7.25,7.79,11.25,17.4
Assault,45.0,109.00,159.00,182.18,249.00,879.0
UrbanPop,6.0,53.25,66.00,74.20,77.75,570.0


## Flagged Values

In [3]:
valid_range = {'Murder': (0, 100_000), 'Assault': (0, 100_000), 'UrbanPop': (0, 100)}

flags = []
for col, (lo, hi) in valid_range.items():
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    for state, val in s.items():
        out_of_range = val < lo or val > hi
        beyond_fence = val < q1 - 1.5 * iqr or val > q3 + 1.5 * iqr
        if out_of_range or beyond_fence:
            flags.append({'State': state, 'Column': col, 'Value': val,
                          'Outside valid range': out_of_range,
                          'Outside IQR fence': beyond_fence})

pd.DataFrame(flags)

,State,Column,Value,Outside valid range,Outside IQR fence
0,South Carolina,Assault,879.0,False,True
1,Iowa,UrbanPop,570.0,True,True
2,New York,UrbanPop,6.0,False,True


In [4]:
# Context for each flag
print(df.loc[['Iowa', 'Nebraska', 'Kansas', 'Minnesota']], '\n')
print(df.loc[['New York', 'Connecticut', 'New Jersey', 'Massachusetts', 'Rhode Island']], '\n')

ratio = df['Assault'] / df['Murder']
print('Next-highest Assault:', df['Assault'].drop('South Carolina').max())
print('SC Murder rank:', int(df['Murder'].rank(ascending=False)['South Carolina']))
print(f'SC Assault/Murder ratio: {ratio["South Carolina"]:.1f}   Median state: {ratio.median():.1f}')

           Murder  Assault  UrbanPop
State                               
Iowa          2.2     56.0       570
Nebraska      4.3    102.0        62
Kansas        6.0    115.0        66
Minnesota     2.7     72.0        66 

               Murder  Assault  UrbanPop
State                                   
New York         11.1    254.0         6
Connecticut       3.3    110.0        77
New Jersey        7.4    159.0        89
Massachusetts     4.4    149.0        85
Rhode Island      3.4    174.0        87 

Next-highest Assault: 337.0
SC Murder rank: 5
SC Assault/Murder ratio: 61.0   Median state: 22.9


## Verdicts

**Flag 1: Iowa, UrbanPop = 570 → IMPOSSIBLE.**

The percentage of the state’s population living in urban areas cannot be greater than 100%, therefore regardless of what else the numbers are for that particular record, the UrbanPop value will fail the logical-range test. The 570 is a possible urban population percentage of 57% with a zero added to the end. Also, 57% sits right beside Iowa's Midwest neighbors, with Nebraska at 62%, Kansas at 66%, and Minnesota at 66%. **Action:** changed to 57; verified using the original published dataset.

**Flag 2: New York, UrbanPop = 6 → NOT impossible, but improbable (extreme low outlier).**

Six percent is a valid urban population percentage, thus it passes the range test. However, it fails the IQR fence. Context also rules it out since New York contains the country's largest city, while every other highly urban Northeastern state falls within a range of 77% to 89%. A single dropped digit from 86 explains the error. **Action:** corrected to 86; verified using the original source. Without a source, I would have converted it to NaN and then imputed rather than keeping a value I believe is incorrect.

**Flag 3: South Carolina, Assault = 879 → NOT impossible, but extremely unusual and probably a data entry error.**

With 879 arrests per 100,000, which is under 1% of residents, it is theoretically possible. Yet, 879 is 2.6x higher than North Carolina's Assault rate, which is the next-highest at 337. While South Carolina's murder rate of 14.4 ranks 5th and is high, it is still not exceptional. Therefore, South Carolina's Assault-to-murder ratio of 61 is about 2.7x the median state's ratio of 22.9. One changed digit, 879 instead of 279, explains the error. **Action:** corrected to 279; verified with the original source. Had the original source confirmed 879 as correct, I would have left it intact because a real extreme value is information, not noise.

**Missing value: Georgia, Assault = NaN.** Filled with 211, the value in the original published dataset (R's `datasets::USArrests`).

In [5]:
df.loc['Iowa', 'UrbanPop'] = 57
df.loc['New York', 'UrbanPop'] = 86
df.loc['South Carolina', 'Assault'] = 279
df.loc['Georgia', 'Assault'] = 211

print('Missing values:', df.isna().sum().sum())
df.describe().T[['min', '25%', '50%', 'mean', '75%', 'max']].round(2)

Missing values: 0


,min,25%,50%,mean,75%,max
Murder,0.8,4.08,7.25,7.79,11.25,17.4
Assault,45.0,109.00,159.00,170.76,249.00,337.0
UrbanPop,32.0,54.50,66.00,65.54,77.75,91.0


## Urban Population Categories

In [6]:
labels = ['small (<50%)', 'medium (50-59%)', 'large (60-69%)', 'extra-large (70%+)']
df['UrbanCategory'] = pd.cut(df['UrbanPop'], bins=[0, 50, 60, 70, 101], labels=labels, right=False)

df.groupby('UrbanCategory', observed=True).agg(
    states=('UrbanPop', 'size'),
    avg_murder=('Murder', 'mean'),
    avg_assault=('Assault', 'mean')).round(1)

,states,avg_murder,avg_assault
UrbanCategory,,,
small (<50%),8,8.2,174.8
medium (50-59%),9,6.7,127.6
large (60-69%),12,7.8,153.5
extra-large (70%+),21,8.1,197.6


## Cleaning Log

| Change | Before | After | Reason |
|--------|--------|-------|--------|
| Iowa UrbanPop | 570 | 57 | Impossible: percent cannot exceed 100 |
| New York UrbanPop | 6 | 86 | Improbable low outlier; verified against source |
| South Carolina Assault | 879 | 279 | Extreme outlier, likely entry error; verified against source |
| Georgia Assault | NaN | 211 | Missing; filled from source |
| Murder | — | — | Screened; no flags |
| UrbanCategory | — | new column | Textbook part c |

## AI Disclosure

**AI Disclosure:** Claude (Anthropic) was used to help explain concepts and draft initial code for this lab. All code was reviewed, retyped, and understood by me. I can explain how `describe()` and the IQR fence rule flag suspicious values, why Iowa's 570 is impossible while New York's 6 and South Carolina's 879 are only extreme, and how `pd.cut` groups states into urban population categories.